# Final Code for Big DataSet

In [6]:
import os
import gzip
import json
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. SETUP PATHS
# Prefer the plain .jsonl in the repository; fallback to .jsonl.gz if present
PREFERRED_PATH = os.path.join('..', 'data', 'raw', 'candidates.jsonl')
ALT_PATH = os.path.join('..', 'data', 'raw', 'candidates.jsonl.gz')
if os.path.exists(PREFERRED_PATH):
    LARGE_DATA_PATH = PREFERRED_PATH
elif os.path.exists(ALT_PATH):
    LARGE_DATA_PATH = ALT_PATH
else:
    raw_dir = os.path.join('..', 'data', 'raw')
    try:
        available = os.listdir(raw_dir)
    except Exception:
        available = []
    raise FileNotFoundError(
        f"Cannot find candidates.jsonl or candidates.jsonl.gz in {raw_dir}. "
        f"Available files: {available}"
    )

OUTPUT_CSV_PATH = '../data/final/submission.csv'

JOB_DESCRIPTION_TEXT = """
Senior AI Engineer Founding Team. Deep technical depth in modern ML systems: 
embeddings, retrieval, ranking, LLMs, fine-tuning. Backend data hybrid: Python, SQL, Spark, Airflow, infrastructure.
"""
TARGET_CITIES = ['noida', 'pune']

# 2. HELPER LOGIC FUNCTIONS
def passes_hard_filters(candidate):
    profile = candidate.get('profile', {}) or {}
    try:
        yoe = float(profile.get('years_of_experience', 0) or 0)
    except Exception:
        yoe = 0
    if not (5 <= yoe <= 9):
        return False

    loc = str(profile.get('location', '')).strip().lower()
    signals = candidate.get('redrob_signals', {}) or {}
    willing_relocate = (candidate.get('willing_to_relocate', False) or 
                        signals.get('willing_to_relocate', False))

    if loc not in TARGET_CITIES and not willing_relocate:
        return False

    title = str(profile.get('current_title', '')).lower()
    non_tech_keywords = ['marketing', 'sales', 'hr', 'recruiter', 'content writer', 'social media']
    skills = candidate.get('skills', []) or []
    if any(kw in title for kw in non_tech_keywords) and len(skills) > 15:
        return False
    return True

def calculate_behavioral_score(candidate):
    signals = candidate.get('redrob_signals', {}) or {}
    if not signals: return 0.5
    score = 1.0
    if signals.get('recruiter_response_rate', 1.0) < 0.30: score -= 0.3
    if signals.get('profile_completeness_score', 100.0) < 50.0: score -= 0.2
    if signals.get('notice_period_days', 0) > 60: score -= 0.1
    return max(0.0, min(1.0, score))

def tech_stack_booster(row):
    # Extract skill names safely
    skills_data = row.get('skills', []) or []
    skills = [s.get('name', '').lower() for s in skills_data]
    must_haves = ['airflow', 'spark', 'python', 'llm', 'embeddings', 'ranking']
    matches = sum(1 for skill in must_haves if skill in skills)
    return matches / len(must_haves) if len(must_haves) > 0 else 0

# 3. STREAM AND FILTER
print("Processing candidates...")
surviving_candidates = []
open_func = gzip.open if LARGE_DATA_PATH.endswith('.gz') else open

with open_func(LARGE_DATA_PATH, 'rt', encoding='utf-8', errors='replace') as fh:
    for raw in fh:
        line = raw.strip()
        if not line: continue
        try:
            candidate = json.loads(line)
            if passes_hard_filters(candidate):
                candidate['behavioral_score'] = calculate_behavioral_score(candidate)
                surviving_candidates.append(candidate)
        except Exception:
            continue

# ensure we have a dataframe even if empty
df_large = pd.DataFrame(surviving_candidates)

# 4. TEXT MATCHING
def extract_text(row):
    profile = row.get('profile', {})
    return f"{profile.get('headline', '')} {profile.get('summary', '')}"

if not df_large.empty:
    df_large['combined_text'] = df_large.apply(extract_text, axis=1)
    vec = TfidfVectorizer(stop_words='english')
    tfidf_matrix = vec.fit_transform([JOB_DESCRIPTION_TEXT] + df_large['combined_text'].tolist())
    # Calculate similarity to the first document (the JD)
    df_large['text_match_score'] = [cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[i+1:i+2])[0][0] for i in range(len(df_large))]
else:
    df_large['combined_text'] = []
    df_large['text_match_score'] = []

# 5. AGGREGATION & TECH BOOSTER
if not df_large.empty:
    df_large['tech_boost'] = df_large.apply(tech_stack_booster, axis=1)
else:
    df_large['tech_boost'] = []

# FINAL FORMULA
if 'behavioral_score' not in df_large.columns:
    df_large['behavioral_score'] = 0.5

df_large['score'] = (df_large.get('text_match_score', 0.0) * 0.4) + \
                    (df_large.get('tech_boost', 0.0) * 0.3) + \
                    (df_large['behavioral_score'] * 0.3)

# 6. SORT WITH TIE-BREAKER
# Sorting by score (DESC) and candidate_id (ASC) solves your validation error!
if 'candidate_id' in df_large.columns:
    df_sorted = df_large.sort_values(by=['score', 'candidate_id'], ascending=[False, True]).reset_index(drop=True)
else:
    df_sorted = df_large.sort_values(by='score', ascending=False).reset_index(drop=True)

# 7. FINAL FORMATTING
df_top100 = df_sorted.head(100).copy()
if len(df_top100) > 0:
    df_top100['rank'] = range(1, len(df_top100) + 1)
else:
    df_top100['rank'] = []

df_top100['reasoning'] = df_top100.apply(lambda row: f"Strong match with {row.get('tech_boost',0)*100:.0f}% tech stack alignment and high behavioral engagement.", axis=1) if not df_top100.empty else []

final_submission = df_top100[['candidate_id', 'rank', 'score', 'reasoning']] if not df_top100.empty else pd.DataFrame(columns=['candidate_id', 'rank', 'score', 'reasoning'])
os.makedirs(os.path.dirname(OUTPUT_CSV_PATH), exist_ok=True)
final_submission.to_csv(OUTPUT_CSV_PATH, index=False)

print("SUCCESS! Submission ready.")

Processing candidates...
SUCCESS! Submission ready.
